# Explore a capture (tinker)

Optional starting point when the workflow notebooks give you an **answer** but you want to **dig deeper** — filter endpoints, replot timelines, combine sidecars, or prototype new checks before adding them to `commonlib/`.

**Guided path:** `whatisit.ipynb` → `*/hang-analysis` or `*/performance-analysis` → Export report → [`HOW-TO-READ-RESULTS.md`](../HOW-TO-READ-RESULTS.md).

**This notebook:** load whatever sidecars exist next to `TERRAFORM_LOG_PATH` and experiment in new cells below.

```bash
export TERRAFORM_LOG_PATH=/path/to/plan-tflog.log
cd notebooks && jupyter lab explore-capture.ipynb
```

In [ ]:
import sys
from pathlib import Path

for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "commonlib" / "config.py").is_file():
        _notebooks_root = _root
        break
    if (_root / "notebooks" / "commonlib" / "config.py").is_file():
        _notebooks_root = _root / "notebooks"
        break
else:
    raise RuntimeError(
        "Could not find notebooks/commonlib/. Start Jupyter from notebooks/ "
        "or open a notebook under export/, plan/, apply/, sdk-plan/, or the notebooks root."
    )

if str(_notebooks_root) not in sys.path:
    sys.path.insert(0, str(_notebooks_root))

from commonlib.notebook_setup import setup

setup()

import pandas as pd
import commonlib.config as cfg
import commonlib.classify_tf_log as classify_tf_log
import commonlib.prep_hang_data as hang
import commonlib.run_report as run_report
import commonlib.normalized_cache as norm_cache

In [ ]:
c = cfg.Config()
log_path = c.TERRAFORM_LOG_PATH
if not log_path:
    raise ValueError(
        "Set TERRAFORM_LOG_PATH to your capture file. "
        "Example: export TERRAFORM_LOG_PATH=/path/to/plan-tflog.log"
    )

capture = Path(log_path)
stem = capture.stem
parent = capture.parent

sidecars = {
    "report": parent / f"{stem}-report.json",
    "norm": parent / f"{stem}-norm.json",
    "sdk_norm": parent / f"{stem}-sdk-norm.json",
}

print(f"Capture: {log_path}")
for name, path in sidecars.items():
    status = "found" if path.is_file() else "(not yet — run the matching workflow notebook)"
    print(f"  {name:10} {path}  {status}")

classification = classify_tf_log.classify_file(log_path)
print()
print(f"Format: {classification.primary_log_format}")
print(f"Workflow hints: {classification.verdict}")
print(f"Lines parsed: {classification.parsed_lines:,}")

## Load structured answers (if exported)

`*-report.json` is the shareable summary — `issue_attribution`, SDK tables, hang verdicts. Re-load it here without re-scanning the GB log.

In [ ]:
report_path = sidecars["report"]
if report_path.is_file():
    report = run_report.load_report(str(report_path))
    print("primary_layer:", report.get("summary", {}).get("primary_layer"))
    print("guidance:", (report.get("issue_attribution") or {}).get("guidance"))
    display(pd.json_normalize(report.get("sdk_call_rates") or []).head(10))
else:
    print("No report yet — run Export report in hang/performance-analysis for this capture.")

## Rescan or slice the capture

Hang scan is the single-pass parser behind verdicts, SDK timelines, and `issue_attribution`. Tweak `TAIL_MINUTES`, `min_count`, or filter the dataframes below.

In [ ]:
TAIL_MINUTES = 5
WORKFLOW = "plan"  # plan | export | apply — filters verdict categories

classification, records, counters = hang.load_hang_scan(log_path, tail_minutes=TAIL_MINUTES)
summary = hang.hang_summary_for_workflow(
    counters, TAIL_MINUTES, WORKFLOW, classification
)
attribution = hang.build_issue_attribution(counters, summary, WORKFLOW)
hang.display_issue_attribution(attribution)

df_rates = hang.sdk_call_rates_dataframe(
    counters, duration_minutes=summary.get("duration_minutes")
)
df_timeline = hang.sdk_timeline_dataframe(counters, bucket_minutes=1.0, top_endpoints=8)
print(f"\nSDK endpoints: {len(df_rates)}  timeline rows: {len(df_timeline)}")

## Example filters (copy and adapt)

Add cells below. Helpers live in `commonlib/` — see **Shared library → Tinker cookbook** in the repo `README.md`.

In [ ]:
# DNC export polling only
if not df_rates.empty:
    display(df_rates[df_rates["is_dnclist_export"]].head(15))

# Timeline for one endpoint (pick from df_rates.method_url)
if not df_timeline.empty:
    top_url = df_rates.iloc[0]["method_url"] if not df_rates.empty else None
    if top_url:
        slice_df = df_timeline[df_timeline["method_url"] == top_url]
        display(slice_df.head(20))

## Normalized cache / SDK pairs

Workflow notebooks write `{stem}-norm.json`. `sdk-analysis.ipynb` writes `{stem}-sdk-norm.json`. Load with pandas or the `prep_*_data.load_normalized_records()` helpers from the matching workflow module.

In [ ]:
for label, path in [("norm", sidecars["norm"]), ("sdk_norm", sidecars["sdk_norm"])]:
    if path.is_file():
        rows = norm_cache.load_cache(str(path))
        print(f"{label}: {len(rows):,} records")
        display(pd.json_normalize(rows).head(3))
    else:
        print(f"{label}: not present")